In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
import joblib
import time # Thư viện để bấm giờ

# ==============================================================================
# 0. CẤU HÌNH GPU (QUAN TRỌNG CHO GPU XỊN)
# ==============================================================================
print("⚙️ Đang kiểm tra GPU...")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        # Cho phép GPU lấy bộ nhớ từ từ thay vì chiếm hết 100% ngay lúc đầu
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"🚀 Đã kích hoạt {len(gpus)} GPU Siêu Khủng: {gpus}")
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ Không tìm thấy GPU. Đang chạy bằng CPU (sẽ chậm hơn).")

# ==============================================================================
# 1. LOAD DỮ LIỆU
# ==============================================================================
try:
    print("\n🔄 Đang load dữ liệu...")
    data = np.load('processed_data.npz')
    X_train = data['X_train']
    y_train = data['y_train']
    X_test  = data['X_test']
    y_test  = data['y_test']
    print(f"✅ Load xong! Tổng mẫu train: {len(X_train)}")
except FileNotFoundError:
    print("❌ Lỗi: Thiếu file 'processed_data.npz'.")
    exit()

# ==============================================================================
# 2. XÂY DỰNG MODEL (CUDNN OPTIMIZED)
# ==============================================================================
# Lưu ý: Để GPU chạy nhanh nhất (dùng CuDNN kernel), cần giữ các tham số mặc định:
# activation='tanh', recurrent_activation='sigmoid', recurrent_dropout=0, unroll=False
# Nếu bạn chỉnh mấy cái này, GPU sẽ chạy chậm lại về tốc độ CPU đó!

model = Sequential()

# Dùng Input layer tường minh để tránh Warning của Keras mới
model.add(Input(shape=(X_train.shape[1], X_train.shape[2])))

# Layer 1
model.add(LSTM(units=128, return_sequences=True)) # Tăng số neuron lên 128 cho máu
model.add(Dropout(0.2))

# Layer 2
model.add(LSTM(units=64, return_sequences=False))
model.add(Dropout(0.2))

# Output
model.add(Dense(units=1))

model.compile(optimizer='adam', loss='mean_squared_error')

# ==============================================================================
# 3. HUẤN LUYỆN (BẤM GIỜ & TỐI ƯU BATCH SIZE)
# ==============================================================================
checkpoint = ModelCheckpoint('best_aqi_model.keras', monitor='val_loss', save_best_only=True, verbose=0)
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# --- TÙY CHỈNH CHO GPU MẠNH ---
# Batch size càng lớn, GPU chạy càng nhanh (nhưng tốn VRAM)
# Với GPU xịn (8GB+ VRAM), hãy để 128, 256 hoặc 512.
BATCH_SIZE = 128 
EPOCHS = 100

print(f"\n🏎️ Bắt đầu đua! Batch Size: {BATCH_SIZE}")
start_time = time.time() # Bắt đầu bấm giờ

history = model.fit(
    X_train, y_train,
    epochs=EPOCHS,
    batch_size=BATCH_SIZE, # Quan trọng nhất để tối ưu GPU
    validation_data=(X_test, y_test),
    callbacks=[checkpoint, early_stopping],
    verbose=1
)

end_time = time.time() # Kết thúc bấm giờ
training_time = end_time - start_time

# ==============================================================================
# 4. KẾT QUẢ
# ==============================================================================
print("-" * 50)
print(f"🏁 HOÀN TẤT HUẤN LUYỆN!")
print(f"⏱️ Tổng thời gian chạy: {training_time:.2f} giây")
if len(history.epoch) > 0:
    print(f"⚡ Trung bình mỗi Epoch: {training_time / len(history.epoch):.4f} giây")
print("-" * 50)

model.save('final_aqi_model.keras')

# (Phần code vẽ biểu đồ giữ nguyên như cũ...)